In [2]:
from pathlib import Path
import warnings

import lightning.pytorch as pl
import terratorch
import torch
from terratorch.registry import BACKBONE_REGISTRY

import loader

pl.seed_everything(64)
warnings.filterwarnings('ignore')

Seed set to 64


In [3]:
data_path = Path('data/zarrs')
rtc_path = data_path / 'swathID_1507_swathDate_2020-07-06_S1RTC.zarr.zip'
s2_path = data_path / 'swathID_1507_swathDate_2020-07-06_S2L2A.zarr.zip'
label_path = data_path / 'swathID_1507_swathDate_2020-07-06.zarr.zip'

In [4]:
# Custom data module for loading our severe weather data
datamodule = loader.SatChipDataModule(batch_size=8, label_path=label_path, s2_path=s2_path, rtc_path=rtc_path)

In [5]:
datamodule.setup('fit')

In [6]:
# checking datasets validation split size
train_dataset = datamodule.train_dataset
print(f'Length of train dataset: {len(train_dataset)}')

val_dataset = datamodule.val_dataset
print(f'Length of validation dataset: {len(val_dataset)}')

Length of train dataset: 392
Length of validation dataset: 98


In [7]:
train_dataset.dataset.plot(train_dataset[0])

AttributeError: 'Tensor' object has no attribute 'astype'

In [13]:
model = BACKBONE_REGISTRY.build(
    'terramind_v1_base',
    modalities=["S2L2A", "S1RTC"],
    pretrained=True,
)

In [17]:
checkpoint_callback = pl.callbacks.ModelCheckpoint(
    dirpath='output/terramind_base_hwds/checkpoints/',
    mode='max',
    monitor='val/mIoU',
    filename='best-mIoU',
    save_weights_only=True,
)

trainer = pl.Trainer(
    accelerator='auto',
    strategy='auto',
    num_nodes=1,
    logger=True,
    max_epochs=2,
    log_every_n_steps=1,
    callbacks=[checkpoint_callback, pl.callbacks.RichProgressBar()],
    default_root_dir='output/terramind_base_hwds/',
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [19]:
model = terratorch.tasks.SemanticSegmentationTask(
    model_factory='EncoderDecoderFactory',  # Combines a backbone with necks, the decoder, and a head
    model_args={
        # TerraMind backbone
        'backbone': 'terramind_v1_base',  # large version: terramind_v1_large
        'backbone_pretrained': True,
        'backbone_modalities': ['S2L2A', 'S1RTC'],
        # Optionally, define the input bands. This is only needed if you select a subset of the pre-training bands, as explained above.
        # "backbone_bands": {"S1GRD": ["VV"]},
        # Necks
        'necks': [
            {
                'name': 'SelectIndices',
                'indices': [2, 5, 8, 11],  # indices for terramind_v1_base
                # "indices": [5, 11, 17, 23] # indices for terramind_v1_large
            },
            {
                'name': 'ReshapeTokensToImage',
                'remove_cls_token': False,
            },  # TerraMind is trained without CLS token, which neads to be specified.
            {
                'name': 'LearnedInterpolateToPyramidal'
            },  # Some decoders like UNet or UperNet expect hierarchical features. Therefore, we need to learn a upsampling for the intermediate embedding layers when using a ViT like TerraMind.
        ],
        # Decoder
        'decoder': 'UNetDecoder',
        'decoder_channels': [512, 256, 128, 64],
        # Head
        'head_dropout': 0.1,
        'num_classes': 2,
    },
    loss='dice',  # We recommend dice for binary tasks and ce for tasks with multiple classes.
    optimizer='AdamW',
    lr=2e-5,  # The optimal learning rate varies between datasets, we recommend testing different once between 1e-5 and 1e-4. You can perform hyperparameter optimization using terratorch-iterate.
    ignore_index=-1,
    freeze_backbone=True,  # Only used to speed up fine-tuning in this demo, we highly recommend fine-tuning the backbone for the best performance.
    freeze_decoder=False,  # Should be false in most cases as the decoder is randomly initialized.
    plot_on_val=False,  # Plot predictions during validation steps
    class_names=['no-damage', 'damage'],  # optionally define class names
)

In [20]:
trainer.fit(model, datamodule=datamodule)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type             ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ model         │ PixelWiseModel   │  103 M │ train │
│ 1 │ criterion     │ DiceLoss         │      0 │ train │
│ 2 │ train_metrics │ MetricCollection │      0 │ train │
│ 3 │ val_metrics   │ MetricCollection │      0 │ train │
│ 4 │ test_metrics  │ ModuleList       │      0 │ train │
└───┴───────────────┴──────────────────┴────────┴───────┘

Trainable params: 15.5 M                                                                                           
Non-trainable params: 87.7 M                                                                                       
Total params: 103 M                                                                                                
Total estimated model params size (MB): 412                                                                        
Modules in train mode: 287                                                                                         
Modules in eval mode: 0

Output()

`Trainer.fit` stopped: `max_epochs=2` reached.
